# Data Literacy Workshop: Day 1 — Practice Notebook 2
## Data-Quality Drill: Validation, Normalization & Deduplication

### Overview
* This is a **focused drill**, not a repeat of the full pipeline. It targets the exercises learners find hardest in the main workshop: regex-based validation, coalescing, MCC/currency normalization, and deduplication.
* Data acquisition and merging are **provided and pre-solved** below so you can jump straight into the cleaning logic — this notebook does not need the mock API server or MongoDB Atlas.
* It reuses the same **Practice dataset** (`DataSet/Practice/`) as `PracticeNotebook_1.ipynb`, so the values you see here differ from the workshop notebook.

## Setup (provided — not an exercise)
Run the next two cells as-is. They load the three practice sources, merge them on `tran_id_raw`, and build `df_canonical` using a private coalescing helper (`_coalesce`) so this setup does not depend on the function you will write in Exercise D1.

In [ ]:
from pathlib import Path
from typing import Any
import re
import pandas as pd

# Resolve path whether notebook is run from project root or PracticeNotebooks/
_nb_dir = Path.cwd()
DATA_DIR = (_nb_dir / "DataSet" / "Practice") if (_nb_dir / "DataSet" / "Practice").exists() \
    else (_nb_dir.parent / "DataSet" / "Practice")

df_settlement = pd.read_csv(DATA_DIR / "settlement.csv", dtype=str, keep_default_na=False, na_values=[])
df_api = pd.read_csv(DATA_DIR / "api.csv", dtype=str, keep_default_na=False, na_values=[])
df_mongo = pd.read_json(DATA_DIR / "mongo.jsonl", lines=True, dtype=str)

print("settlement:", df_settlement.shape, "| api:", df_api.shape, "| mongo:", df_mongo.shape)

In [ ]:
CANONICAL_COLS = [
    "cpd_mnth_id","tran_id_raw","pymt_crd_acct_num","ntwrk_id","issr_ctry_cd","issr_regn_cd",
    "issr_usr_bus_id","lgl_ntwrk_grp_cd","acqr_bin_ctry_cd","acqr_bin_regn_cd","tran_curr_cd",
    "eci_moto_cd","mrch_catg_cd","mrch_nm_raw","mrch_ctry_cd","mrch_ctry_alpha_cd","mrch_regn_cd",
    "acct_prod_id","prod_brnd_cd","prod_typ_cd","crd_id_mthd_cd","pos_entry_mode_cd","pos_env_cd",
    "pos_terml_cpbty_cd","pin_entry_cpbty_cd","auth_resp_cd"
]

merged = df_settlement.merge(df_api, on="tran_id_raw", how="left", suffixes=("_set", "_api"))
if "tran_id_raw" in df_mongo.columns:
    merged = merged.merge(df_mongo.drop_duplicates(subset=["tran_id_raw"]), on="tran_id_raw", how="left", suffixes=("", "_mongo"))

def _coalesce(df, base_name):
    """Private helper used only for this notebook's setup — not the exercise you will write."""
    set_col, api_col, mongo_col = f"{base_name}_set", f"{base_name}_api", base_name
    out = pd.Series(pd.NA, index=df.index, dtype="object")
    if set_col in df.columns:
        out = df[set_col].where(df[set_col].notna(), out)
    if api_col in df.columns:
        out = out.where(out.notna(), df[api_col])
    if mongo_col in df.columns:
        out = out.where(out.notna(), df[mongo_col])
    return out

df_canonical = pd.DataFrame(index=merged.index)
for _col in CANONICAL_COLS:
    try:
        df_canonical[_col] = _coalesce(merged, _col)
    except Exception:
        df_canonical[_col] = merged.get(_col, pd.Series(pd.NA, index=merged.index))

print("df_canonical ready:", df_canonical.shape)

## Exercise D1: Re-implement `coalesce_columns_vectorized`
Write your own version of the coalescing helper from the main workshop (Exercise 5.2), and verify it produces the exact same `tran_curr_cd` column as the pre-built `df_canonical` above.

In [ ]:
def coalesce_columns_vectorized(df: pd.DataFrame, base_name: str) -> pd.Series:
    """
    Combine the same field coming from 3 different datasets (Settlement, API, Mongo) into one
    canonical column using the priority: Settlement > API > Mongo.
    """
    # Steps:
    #   1. Work out the expected column names: f"{base_name}_set", f"{base_name}_api",
    #      and the plain base_name (for Mongo)
    #   2. Start with an all-NA Series aligned to df's index
    #   3. If the settlement column exists, fill from it wherever the running result is
    #      still NA (Series.where)
    #   4. Do the same for the API column, then the Mongo column, always only filling
    #      positions that are still NA
    #   5. Return the final coalesced Series
    # TODO: implement
    pass 


In [ ]:
try:
    # Checkpoint: your coalesce function should match the pre-built canonical column exactly.
    _your_result = coalesce_columns_vectorized(merged, "tran_curr_cd")
    assert _your_result.equals(df_canonical["tran_curr_cd"]), (
        "Your coalesce_columns_vectorized output does not match the pre-built df_canonical column "
        "for 'tran_curr_cd' — check your Settlement > API > Mongo precedence."
    )
    print("Checkpoint OK — your coalesce function matches the reference output.")
except Exception as _e:
    print(f'\u26a0\ufe0f  Checkpoint skipped (not yet implemented): {_e}')


## Exercise D2: `extract_code`
Extract the leading numeric code from a labeled field like `"5411 - Grocery Stores"`.

In [ ]:
def extract_code(series: pd.Series) -> pd.Series:
    """
    Extracts the leading code from a labeled string field in the format "NNN - Label".
    - If the separator " - " is present, returns the code portion before it.
    - If not, returns the stripped original value.
    """
    # Steps:
    #   1. Fill NA with "" and strip whitespace from every value
    #   2. Identify which rows contain the " - " separator
    #   3. For rows with the separator, split on " - " once and keep the left-hand part, stripped
    #   4. For rows without the separator (and not empty), keep the value as-is
    #   5. Return the resulting Series
    # TODO: implement
    pass 


## Exercise D3: `pan_valid_mask`
A Boolean mask for the masked PAN format `XXXXXXXXXXXX1234` (12 X's + 4 digits).

In [ ]:
def pan_valid_mask(series: pd.Series) -> pd.Series:
    """
    Returns a Boolean mask: True where the value matches 'XXXXXXXXXXXX1234'
    (exactly 12 'X' characters followed by exactly 4 digits).
    """
    # Steps:
    #   1. Fill NA with "" and cast to string
    #   2. Use a regex to check the value matches exactly 12 'X' characters followed by
    #      exactly 4 digits (str.match with na=False)
    # TODO: implement
    pass 


## Exercise D4: `tid_valid_mask`
A Boolean mask for a valid 15-digit transaction ID.

In [ ]:
def tid_valid_mask(series: pd.Series) -> pd.Series:
    """
    Returns a Boolean mask: True where the value is exactly 15 numeric digits.
    """
    # Steps:
    #   1. Fill NA with "" and cast to string
    #   2. Use a regex to check the value is exactly 15 numeric digits (str.match with na=False)
    # TODO: implement
    pass 


## Exercise D5: Apply the validity flags
Use D2–D4 to add `is_pan_valid`, `is_tid_valid`, and `is_mcc_valid` to `df_canonical` (an MCC is valid when its extracted code is exactly 4 digits).

In [ ]:
# TODO:
#   1. df_canonical["is_pan_valid"] = pan_valid_mask(df_canonical["pymt_crd_acct_num"])
#   2. df_canonical["is_tid_valid"] = tid_valid_mask(df_canonical["tran_id_raw"])
#   3. Use extract_code on df_canonical["mrch_catg_cd"] to get the MCC's numeric code,
#      then set df_canonical["is_mcc_valid"] to True where that code is exactly 4 digits

In [ ]:
try:
    for _col in ("is_pan_valid", "is_tid_valid", "is_mcc_valid"):
        assert _col in df_canonical.columns, f"'{_col}' is missing — check Exercise D5."
    print("Checkpoint OK — validity flags present:",
          df_canonical[["is_pan_valid", "is_tid_valid", "is_mcc_valid"]].mean().round(3).to_dict())
except Exception as _e:
    print(f'\u26a0\ufe0f  Checkpoint skipped (not yet implemented): {_e}')


## Exercise D6: `normalize_mcc_val`
Restore MCC values to the canonical `"NNNN - Label"` form where possible (Exercise 6.6.1 in the main workshop).

In [ ]:
def normalize_mcc_val(val: Any) -> Any:
    """
    Normalize a merchant category code (MCC) value into the canonical "NNNN - Label" form
    where possible; otherwise return pd.NA.
    """
    # Steps:
    #   1. If val is missing (pd.isna), return pd.NA
    #   2. Strip the value to a string; if it already matches "NNNN - " at the start, return as-is
    #   3. Otherwise, try to extract a leading 4-digit code and look it up in a small dict of
    #      known MCC codes -> labels (e.g. "5411": "5411 - Grocery Stores", "5311": "5311 -
    #      Department Stores", "5542": "5542 - Automated Fuel Dispensers", "5812": "5812 -
    #      Restaurants", "6011": "6011 - ATM Cash", "5992": "5992 - Florists")
    #   4. Return the mapped label if found, else pd.NA
    # TODO: implement
    pass 

# TODO: apply normalize_mcc_val to df_canonical["mrch_catg_cd"] and refresh is_mcc_valid
# (True wherever the normalized value is non-null)

## Exercise D7: `normalize_currency_code`
Normalize the transaction currency into a clean 3-digit numeric code (Exercise 6.7.1 in the main workshop).

In [ ]:
def normalize_currency_code(val):
    """
    Normalize a transaction currency code into a clean 3-digit numeric string;
    otherwise return pd.NA.
    """
    # Steps:
    #   1. If val is missing (pd.isna), return pd.NA
    #   2. If the value contains " - ", split on it and keep the left-hand part only if it's
    #      exactly 3 digits, else return pd.NA
    #   3. If there's no " - ", check whether the whole value is already exactly 3 digits;
    #      if so return it, otherwise return pd.NA
    # TODO: implement
    pass 

# TODO: apply normalize_currency_code to df_canonical["tran_curr_cd"], storing the result in
# "_tran_curr_code" and a "_tran_curr_valid" Boolean flag (True where that result is non-null)

## Exercise D8: Deduplicate by `tran_id_raw`
Drop duplicate transaction IDs, keeping the first occurrence, and report how many were dropped.

In [ ]:
# TODO:
#   1. Build a Boolean mask of duplicate tran_id_raw values, keeping the first occurrence
#      (df_canonical["tran_id_raw"].duplicated(keep="first"))
#   2. Save the duplicate rows separately in `dropped_dupes` for reference
#   3. Keep only the non-duplicate rows in df_canonical and reset its index

In [ ]:
try:
    assert df_canonical["tran_id_raw"].duplicated().sum() == 0, (
        "df_canonical still has duplicate tran_id_raw values — check Exercise D8."
    )
    print(f"Checkpoint OK — no duplicate tran_id_raw remain ({len(df_canonical)} rows).")
except Exception as _e:
    print(f'\u26a0\ufe0f  Checkpoint skipped (not yet implemented): {_e}')


## Summary
* You re-implemented the coalescing helper and verified it against a known-good reference output.
* You rebuilt the PAN, TID, and MCC validity checks from scratch.
* You normalized MCC and currency codes and deduplicated the canonical dataset — the same skills as Activities 4 and 6 of the main workshop, drilled in isolation on new data.